# 05 — Build the Master Experimental Dataset
This notebook creates a traceable research database from the two standardized welding datasets and the two literature-derived text sources. It does **not** assume all records are interchangeable for machine learning.

## Inputs required
Upload these two generated files when prompted:
- `conventional_laser_welding_standardized.csv` (27 rows)
- `alcu_wobbling_welding_standardized.csv` (36 rows)

The literature-derived records from `first.txt` and `second..txt` are transcribed below from the source tables already collected. Check them against the papers before publication. Source 2 force values are retained as reported but flagged because some min/max/average values are internally inconsistent.


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from pathlib import Path
from IPython.display import display

uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))


In [ ]:
# Read the two generated CSV files from this Colab session
conv_name = 'conventional_laser_welding_standardized.csv'
wob_name = 'alcu_wobbling_welding_standardized.csv'
if conv_name not in uploaded or wob_name not in uploaded:
    raise FileNotFoundError('Please upload BOTH standardized CSV files using the upload prompt above. Check the exact filenames.')

conv = pd.read_csv(conv_name)
wob = pd.read_csv(wob_name)
print('Conventional shape:', conv.shape)
print('Wobbling shape:', wob.shape)
display(conv.head(3)); display(wob.head(3))

assert len(conv) == 27, f'Expected 27 conventional records, found {len(conv)}'
assert len(wob) == 36, f'Expected 36 wobbling records, found {len(wob)}'


## Literature source 1 — measured joint outcomes
Only the 11 specimen conditions that have measured outcomes in Table 5 are included as supervised outcome records. Process-only rows from Table 1 are not falsely labelled with outcomes.


In [ ]:
s1 = pd.DataFrame([
    ['23/20',2300,20,115.0,33.4,0.278,0.140],
    ['23/28.6',2300,28.6,80.5,32.8,0.262,0.132],
    ['25/15.4',2500,15.4,162.5,26.7,0.328,0.208],
    ['25/20',2500,20,125.0,32.8,0.309,0.165],
    ['25/28.6',2500,28.6,87.5,33.3,0.285,0.146],
    ['25/40',2500,40,62.5,32.7,0.261,0.129],
    ['27/20',2700,20,135.0,21.5,0.346,0.233],
    ['27/28.6',2700,28.6,94.5,33.7,0.300,0.156],
    ['27/40',2700,40,67.5,32.8,0.280,0.141],
    ['29/20',2900,20,145.0,16.0,0.372,0.248],
    ['29/40',2900,40,72.5,21.7,0.291,0.155],
], columns=['specimen_id','laser_power_W','welding_speed_mm_s','reported_heat_input_J_mm','tensile_shear_strength_MPa','electrical_resistance_microohm','welding_area_mm2'])
s1['source_id'] = 'shin_2021_metals_11050831'
s1['study_id'] = 'study_shin_2021'
s1['process_type'] = 'conventional_laser_welding'
s1['record_id'] = ['S1_' + str(i).zfill(2) for i in range(1, len(s1)+1)]
s1['data_quality_flag'] = 'transcribed_from_measured_outcome_table; verify_against_original_paper'
s1['ml_eligible_initial'] = True
display(s1)


## Literature source 2 — reported tensile-test table
The 10 rows are retained in the research database, but are marked **not initially eligible for model training** because the reported force min/max/average values contain inconsistencies. Missing tests remain missing; no values are repaired or inferred.


In [ ]:
s2 = pd.DataFrame([
    [1,'Al',200,125.8,148.0,138.0,'Al'],
    [2,'Al',400,118.2,152.1,137.3,'Al'],
    [3,'Al',600,129.9,149.9,142.0,'Al'],
    [4,'Al',800,143.6,131.7,137.7,'Joint'],
    [5,'Al',1000,np.nan,np.nan,np.nan,'Not reported'],
    [6,'Cu',200,126.4,133.8,132.8,'Al'],
    [7,'Cu',400,144.4,144.2,151.8,'Al'],
    [8,'Cu',600,150.0,138.6,148.1,'Al'],
    [9,'Cu',800,153.8,142.8,154.4,'Cu'],
    [10,'Cu',1000,np.nan,np.nan,np.nan,'Not reported'],
], columns=['parameter_no','upper_material','welding_speed_mm_s','force_min_N_reported','force_max_N_reported','force_average_N_reported','fracture_area_reported'])
s2['source_id'] = 'lee_2022_metals_12111957'
s2['study_id'] = 'study_lee_2022'
s2['process_type'] = 'laser_welding_upper_material_comparison'
s2['record_id'] = ['S2_' + str(i).zfill(2) for i in range(1, len(s2)+1)]
s2['data_quality_flag'] = 'reported_force_values_require_source_table_verification'
s2['ml_eligible_initial'] = False
s2['min_exceeds_max_flag'] = s2['force_min_N_reported'] > s2['force_max_N_reported']
s2['average_outside_reported_range_flag'] = ((s2['force_average_N_reported'] < s2['force_min_N_reported']) | (s2['force_average_N_reported'] > s2['force_max_N_reported'])) & s2['force_average_N_reported'].notna()
display(s2)
print('Min exceeds max:', int(s2['min_exceeds_max_flag'].sum()))
print('Average outside min/max:', int(s2['average_outside_reported_range_flag'].sum()))


## Build a traceable master table
The master table uses a long/record-oriented structure. Study-specific variables remain as columns and are left blank when not measured. `process_type`, `source_id`, and `study_id` identify the origin and process family. This is a data inventory, not a claim that all rows can be pooled for one model.


In [ ]:
# Add common provenance and eligibility fields to the two standardized experimental datasets
conv = conv.copy(); wob = wob.copy()
conv['process_type'] = 'conventional_laser_welding'
conv['ml_eligible_initial'] = True
conv['data_quality_flag'] = np.where(conv.get('weld_observation_zero_flag', False).astype(bool), 'zero_output_record_requires_physical_interpretation', 'retain_source_measurement')
wob['process_type'] = 'wobble_laser_welding'
wob['ml_eligible_initial'] = True
wob['data_quality_flag'] = np.where(wob.get('width_measurements_missing', False).astype(bool), 'bead_width_missing; penetration_depth_retained', 'retain_source_measurement')

# Keep source-specific columns; concat aligns columns without inventing values.
master = pd.concat([conv, wob, s1, s2], ignore_index=True, sort=False)
master.insert(0, 'master_record_id', ['MASTER_' + str(i).zfill(4) for i in range(1, len(master)+1)])
print('Master shape:', master.shape)
print('Rows by process type:')
display(master.groupby('process_type', dropna=False).size().rename('records').reset_index())
display(master.head(5))


In [ ]:
# Export separate source tables plus a compact data dictionary and audit summary
out = Path('master_outputs'); out.mkdir(exist_ok=True)
master.to_csv(out/'master_experimental_dataset.csv', index=False)
s1.to_csv(out/'source1_measured_outcomes.csv', index=False)
s2.to_csv(out/'source2_force_values_flagged.csv', index=False)

audit = pd.DataFrame([
    ['conventional_laser_welding',len(conv),'27 expected; preserve zero-output flags'],
    ['wobble_laser_welding',len(wob),'36 expected; preserve missing bead widths'],
    ['source1_measured_outcomes',len(s1),'11 matched specimens with measured outcomes'],
    ['source2_force_table',len(s2),'10 rows retained; force values flagged for verification'],
], columns=['dataset_component','record_count','handling_note'])
audit.to_csv(out/'master_dataset_audit.csv', index=False)

data_dictionary = pd.DataFrame([
    ['master_record_id','Unique row identifier generated in this notebook','All'],
    ['source_id','Identifier for original data source/study','All'],
    ['study_id','Study-level grouping identifier; use for grouped validation','All'],
    ['process_type','Process family; do not pool unlike processes without justification','All'],
    ['laser_power_W','Laser power in watts, when reported','Selected studies'],
    ['welding_speed_mm_s','Travel/scan/welding speed in mm/s; confirm definition by study','Selected studies'],
    ['tensile_shear_strength_MPa','Measured tensile shear strength in MPa','Source 1'],
    ['electrical_resistance_microohm','Electrical resistance in micro-ohms','Source 1'],
    ['welding_area_mm2','Welding area in mm^2','Source 1'],
    ['ml_eligible_initial','Initial screening flag only; not final approval for model training','All'],
    ['data_quality_flag','Human-readable caution; never silently repair flagged values','All'],
], columns=['column_name','definition','availability'])
data_dictionary.to_csv(out/'master_data_dictionary.csv', index=False)

print('Created files:')
for p in sorted(out.glob('*.csv')): print(p, p.stat().st_size, 'bytes')
files.download(str(out/'master_experimental_dataset.csv'))
files.download(str(out/'master_dataset_audit.csv'))
files.download(str(out/'master_data_dictionary.csv'))
files.download(str(out/'source1_measured_outcomes.csv'))
files.download(str(out/'source2_force_values_flagged.csv'))


## Important interpretation notes
1. The conventional and wobble datasets have different input variables and remain distinguishable by `process_type`.
2. Conventional records with all-zero weld measurements are retained and flagged, not deleted.
3. Wobble records with missing bead widths are retained; penetration depth may still be usable for a target-specific model.
4. Source 1 contributes 11 measured outcome records only. The other process-only conditions are not assigned invented outcomes.
5. Source 2 force results are retained exactly as transcribed but flagged and excluded from initial ML eligibility pending verification.
6. Before modelling, perform target-specific inclusion checks and use study-grouped validation to reduce leakage.
